# 1 Exercise-1
## 1.1 Data Exploration
### Q.1.1.1

Examine the summary statistics (mean, standard deviation, min, max, quartiles) of
all 81 features and the target. Which features show the highest variation? Justify
why comparing raw standard deviations across these features can be misleading,
and propose a more suitable measure.

In [1]:
from pathlib import Path
import pandas as pd
import zipfile
import urllib.request

def load_superconductivity_data():
    zip_path = Path("datasets/superconductivty+data.zip")
    if not zip_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://archive.ics.uci.edu/static/public/464/superconductivty+data.zip"
        urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(path="datasets/superconductivity")
    return pd.read_csv(Path("datasets/superconductivity/train.csv"))

data = load_superconductivity_data()

data.describe()



,number_of_elements,mean_atomic_mass,wtd_mean_atomic_mass,gmean_atomic_mass,wtd_gmean_atomic_mass,entropy_atomic_mass,wtd_entropy_atomic_mass,range_atomic_mass,wtd_range_atomic_mass,std_atomic_mass,...,wtd_mean_Valence,gmean_Valence,wtd_gmean_Valence,entropy_Valence,wtd_entropy_Valence,range_Valence,wtd_range_Valence,std_Valence,wtd_std_Valence,critical_temp
count,21263.000000,21263.000000,21263.000000,21263.000000,21263.000000,21263.000000,21263.000000,21263.000000,21263.000000,21263.000000,...,21263.000000,21263.000000,21263.000000,21263.000000,21263.000000,21263.000000,21263.000000,21263.000000,21263.000000,21263.000000
mean,4.115224,87.557631,72.988310,71.290627,58.539916,1.165608,1.063884,115.601251,33.225218,44.391893,...,3.153127,3.056536,3.055885,1.295682,1.052841,2.041010,1.483007,0.839342,0.673987,34.421219
std,1.439295,29.676497,33.490406,31.030272,36.651067,0.364930,0.401423,54.626887,26.967752,20.035430,...,1.191249,1.046257,1.174815,0.393155,0.380291,1.242345,0.978176,0.484676,0.455580,34.254362
min,1.000000,6.941000,6.423452,5.320573,1.960849,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000210
25%,3.000000,72.458076,52.143839,58.041225,35.248990,0.966676,0.775363,78.512902,16.824174,32.890369,...,2.116732,2.279705,2.091251,1.060857,0.775678,1.000000,0.921454,0.451754,0.306892,5.365000
50%,4.000000,84.922750,60.696571,66.361592,39.918385,1.199541,1.146783,122.906070,26.636008,45.123500,...,2.618182,2.615321,2.434057,1.368922,1.166532,2.000000,1.063077,0.800000,0.500000,20.000000
75%,5.000000,100.404410,86.103540,78.116681,73.113234,1.444537,1.359418,154.119320,38.356908,59.322812,...,4.026201,3.727919,3.914868,1.589027,1.330801,3.000000,1.918400,1.200000,1.020436,63.000000
max,9.000000,208.980400,208.980400,208.980400,208.980400,1.983797,1.958203,207.972460,205.589910,101.019700,...,7.000000,7.000000,7.000000,2.141963,1.949739,6.000000,6.992200,3.000000,3.000000,185.000000


In [2]:
features = data.drop(columns=['critical_temp'])
# if the 'material' formula column ended up in there too, drop that as well:
# features = data.drop(columns=['critical_temp', 'material'])

desc = features.describe().T

# Q: which features show the highest variation, by raw std?
top_std = desc.sort_values('std', ascending=False)
print(top_std[['mean', 'std']].head(10)) #I can add more, like min, max and etc, but to prove that raw std can be misleading, we will keep these two

bottom_std = desc.sort_values('std', ascending=True)
print(bottom_std[['mean', 'std']].head(10))


                          mean          std
range_Density      8665.438818  4097.126831
wtd_gmean_Density  3117.241110  3975.122587
gmean_Density      3460.692235  3703.256370
wtd_mean_Density   5267.188547  3221.314506
mean_Density       6111.465214  2846.785185
wtd_range_Density  2902.736814  2398.471020
std_Density        3416.910784  1673.624915
wtd_std_Density    3319.170628  1611.799629
range_fie           572.222612   309.614442
wtd_range_fie       483.517264   224.042874
                                     mean       std
wtd_entropy_ElectronAffinity     0.770757  0.285986
wtd_entropy_ThermalConductivity  0.539991  0.318248
wtd_entropy_Density              0.856037  0.319761
entropy_ThermalConductivity      0.727630  0.325976
wtd_entropy_fie                  0.926726  0.334018
entropy_Density                  1.072425  0.342356
entropy_ElectronAffinity         1.070250  0.343391
entropy_atomic_mass              1.165608  0.364930
wtd_entropy_FusionHeat           0.914065  0.370

### Answer

Sorting the features by `std` (from `describe()`) shows that all `*_Density` features
have the highest raw standard deviation. This is because density is measured in kg/m³, so its values sit
on a much larger numeric scale. This makes comparing raw standard deviations across features misleading, since a feature's raw std reflects its unit/scale as much as its actual variability.

A more suitable measure is the **coefficient of variation** (CV) (`std / mean`). Accord to Science direct CV is a statistic that is the ratio of the standard deviation to the mean expressed in percentage (https://www.sciencedirect.com/topics/engineering/coefficient-of-variation).
  Which is scale-free and lets us compare relative spread regardless of units. Here is an example of the use of CV.

In [3]:
desc['CV_%'] = (desc['std'] / desc['mean']).abs() * 100   # coefficient of variation
top_cv = desc.sort_values('CV_%', ascending=False)
print(top_cv[['mean', 'std', 'CV_%']].head(10))

bottom_cv = desc.sort_values('CV_%', ascending=True)
print(top_cv[['mean', 'std', 'CV_%']].head(10))

                                      mean          std        CV_%
wtd_gmean_ThermalConductivity    27.308061    40.191150  147.176869
wtd_range_FusionHeat              8.218528    11.414066  138.882120
wtd_gmean_FusionHeat             10.141161    13.134007  129.511871
wtd_gmean_Density              3117.241110  3975.122587  127.520537
gmean_ThermalConductivity        29.841727    34.059581  114.134080
gmean_Density                  3460.692235  3703.256370  107.009122
std_FusionHeat                    8.323333     8.671651  104.184836
wtd_mean_FusionHeat              13.848001    14.279335  103.114779
gmean_FusionHeat                 10.136977    10.065901   99.298842
range_FusionHeat                 21.138994    20.370620   96.365133
                                      mean          std        CV_%
wtd_gmean_ThermalConductivity    27.308061    40.191150  147.176869
wtd_range_FusionHeat              8.218528    11.414066  138.882120
wtd_gmean_FusionHeat             10.141161    13

Ranking by CV instead of raw std gives a completely different result. `wtd_gmean_ThermalConductivity`,
`wtd_range_FusionHeat`, and `wtd_gmean_FusionHeat` now on the top and all density related features do not stand out. This confirms that
Density's high raw std was mostly a scale effect, not genuinely higher relative variability.

## Q.1.1.2

